In [4]:

spark.sql("""
CREATE OR REPLACE TABLE gold.monthly_country_sales
USING DELTA
AS
SELECT
    DATE_FORMAT(InvoiceDate, 'yyyy-MM') AS SalesMonth,
    Country,

    COUNT(*) AS TransactionRows,

    SUM(
        CASE WHEN TransactionClass = 'RegularSale'
        THEN LineAmount ELSE 0 END
    ) AS GrossSales,

    SUM(
        CASE WHEN TransactionClass = 'Cancelled'
        THEN LineAmount ELSE 0 END
    ) AS ReturnsAmount,

    SUM(LineAmount) AS NetSales

FROM silver.retail_transactions
WHERE TransactionClass IN ('RegularSale', 'Cancelled')
GROUP BY
    DATE_FORMAT(InvoiceDate, 'yyyy-MM'),
    Country
""")

print("Gold table created successfully.")



StatementMeta(, febaaa3c-0696-4636-8933-10bbaac1489e, 6, Finished, Available, Finished, False)

Gold table created successfully.


In [5]:

display(spark.sql("""
SELECT
    SUM(TransactionRows) AS TransactionRows,
    ROUND(SUM(GrossSales), 2) AS GrossSales,
    ROUND(SUM(ReturnsAmount), 2) AS ReturnsAmount,
    ROUND(SUM(NetSales), 2) AS NetSales
FROM gold.monthly_country_sales
"""))


StatementMeta(, febaaa3c-0696-4636-8933-10bbaac1489e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78749dce-0de4-4d3f-aed5-fae9a0d17134)

In [1]:

display(spark.sql("""
WITH Comparison AS (

    SELECT
        '01 Bronze' AS Stage,
        COUNT(*) AS RowsIncluded,
        SUM(
            TRY_CAST(Quantity AS DECIMAL(18,6)) *
            TRY_CAST(Price AS DECIMAL(18,6))
        ) AS TotalAmount
    FROM bronze.retail_transactions

    UNION ALL

    SELECT
        '02 Silver',
        COUNT(*),
        SUM(LineAmount)
    FROM silver.retail_transactions

    UNION ALL

    SELECT
        '03 Gold',
        SUM(TransactionRows),
        SUM(NetSales)
    FROM gold.monthly_country_sales
)

SELECT
    Stage,
    RowsIncluded,
    ROUND(TotalAmount, 2) AS TotalAmount
FROM Comparison
ORDER BY Stage
"""))


StatementMeta(, 9a083ca3-75e0-42a6-9377-642545ab7661, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 556554a7-b97e-4e35-b10b-5ce9abdbc9c6)

In [2]:

display(spark.sql("""
WITH SilverTotals AS (
    SELECT
        COUNT(*) AS SilverRows,
        SUM(LineAmount) AS SilverAmount
    FROM silver.retail_transactions
),
GoldTotals AS (
    SELECT
        SUM(TransactionRows) AS GoldRows,
        SUM(NetSales) AS GoldAmount
    FROM gold.monthly_country_sales
),
ExcludedTotals AS (
    SELECT
        COUNT(*) AS ExcludedRows,
        SUM(LineAmount) AS ExcludedAmount
    FROM silver.retail_transactions
    WHERE TransactionClass NOT IN
        ('RegularSale', 'Cancelled')
)
SELECT
    s.SilverRows - g.GoldRows AS RowDifference,
    e.ExcludedRows,
    ROUND(s.SilverAmount - g.GoldAmount, 2)
        AS AmountDifference,
    ROUND(e.ExcludedAmount, 2) AS ExcludedAmount,
    CASE
        WHEN s.SilverRows - g.GoldRows = e.ExcludedRows
         AND ABS(
             (s.SilverAmount - g.GoldAmount)
             - e.ExcludedAmount
         ) < 0.000001
        THEN 'PASS'
        ELSE 'FAIL'
    END AS ReconciliationStatus
FROM SilverTotals s
CROSS JOIN GoldTotals g
CROSS JOIN ExcludedTotals e
"""))


StatementMeta(, 9a083ca3-75e0-42a6-9377-642545ab7661, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8b36586-497c-4c1c-8720-1e29ccc8d9f1)

In [3]:

spark.sql("""
CREATE OR REPLACE TABLE gold.monthly_country_sales_scenario
USING DELTA
AS
SELECT
    DATE_FORMAT(InvoiceDate, 'yyyy-MM') AS SalesMonth,
    Country,
    COUNT(*) AS TransactionRows,
    SUM(LineAmount) AS NetSales
FROM silver.retail_transactions
WHERE TransactionClass IN (
    'RegularSale',
    'Cancelled',
    'BadDebtAdjustment'
)
GROUP BY
    DATE_FORMAT(InvoiceDate, 'yyyy-MM'),
    Country
""")

print("Alternative Gold scenario created.")


StatementMeta(, 9a083ca3-75e0-42a6-9377-642545ab7661, 5, Finished, Available, Finished, False)

Alternative Gold scenario created.


In [4]:

display(spark.sql("""
SELECT
    'Original Gold' AS Scenario,
    SUM(TransactionRows) AS RowsIncluded,
    ROUND(SUM(NetSales), 2) AS NetSales
FROM gold.monthly_country_sales

UNION ALL

SELECT
    'Alternative Gold',
    SUM(TransactionRows),
    ROUND(SUM(NetSales), 2)
FROM gold.monthly_country_sales_scenario
"""))


StatementMeta(, 9a083ca3-75e0-42a6-9377-642545ab7661, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a22136f6-b551-4ed1-afe5-27cb44c8d42e)